In [2]:
import ROOT
import math
import os
from IPython.display import display

%jsroot on

# ============================================================
# SETTINGS
# ============================================================

file_path = "/root/geant4/detector/Tumor/Tumor1/tumor1.root"
tree_name = "t"

selected_volume = 2
selected_process = 2013
selected_pdg = 22

# Incident beam direction: /gps/direction 0 -1 0
incident_direction = (0.0, -1.0, 0.0)

# Set True for logarithmic count/color scale
use_logz = False


# ============================================================
# OPEN FILE
# ============================================================

if not os.path.exists(file_path):
    raise FileNotFoundError(file_path)

f = ROOT.TFile.Open(file_path)

if not f or f.IsZombie():
    raise RuntimeError(f"Could not open {file_path}")

t = f.Get(tree_name)

if not t:
    raise RuntimeError(f"Tree '{tree_name}' was not found")

print("Tree entries:", t.GetEntries())


# ============================================================
# REMOVE OLD OBJECTS
# ============================================================

for name in ["h_de_theta", "h_et_theta", "c_theta"]:
    obj = ROOT.gROOT.FindObject(name)
    if obj:
        obj.Delete()


# ============================================================
# HISTOGRAMS
# ============================================================

h_de_theta = ROOT.TH2F(
    "h_de_theta",
    "de versus scattering angle, vlm=2;"
    "#theta (degrees);de (keV)",
    90, 0, 180,
    140, 0, 700
)

h_et_theta = ROOT.TH2F(
    "h_et_theta",
    "et versus scattering angle, vlm=2;"
    "#theta (degrees);et (keV)",
    90, 0, 180,
    140, 0, 700
)

h_de_theta.SetStats(0)
h_et_theta.SetStats(0)


# ============================================================
# INCIDENT DIRECTION
# ============================================================

ix, iy, iz = incident_direction

incident_norm = math.sqrt(ix**2 + iy**2 + iz**2)

ix /= incident_norm
iy /= incident_norm
iz /= incident_norm


# ============================================================
# EVENT LOOP
# ============================================================

selected = 0

for event in t:

    n = min(
        len(event.pdg),
        len(event.pro),
        len(event.vlm),
        len(event.de),
        len(event.et),
        len(event.px),
        len(event.py),
        len(event.pz)
    )

    for i in range(n):

        if int(event.vlm[i]) != selected_volume:
            continue

        if int(event.pro[i]) != selected_process:
            continue

        if int(event.pdg[i]) != selected_pdg:
            continue

        # Do not require stp==0 initially.
        # Add this only after checking the stp distribution.
        #
        # if int(event.stp[i]) != 0:
        #     continue

        px = float(event.px[i])
        py = float(event.py[i])
        pz = float(event.pz[i])

        p = math.sqrt(px**2 + py**2 + pz**2)

        if p <= 0:
            continue

        ux = px / p
        uy = py / p
        uz = pz / p

        cos_theta = ix*ux + iy*uy + iz*uz
        cos_theta = max(-1.0, min(1.0, cos_theta))

        theta = math.degrees(math.acos(cos_theta))

        de_value = float(event.de[i])
        et_value = float(event.et[i])

        if math.isfinite(de_value):
            h_de_theta.Fill(theta, de_value)

        if math.isfinite(et_value):
            h_et_theta.Fill(theta, et_value)

        selected += 1


print("Selected points:", selected)
print("de entries:", h_de_theta.GetEntries())
print("et entries:", h_et_theta.GetEntries())


# ============================================================
# FIND ORIGINAL MAXIMUM BIN
# ============================================================

max_counts = -1.0
max_bin_x = -1
max_bin_y = -1

for bx in range(1, h_et_theta.GetNbinsX() + 1):
    for by in range(1, h_et_theta.GetNbinsY() + 1):

        counts = h_et_theta.GetBinContent(bx, by)

        if counts > max_counts:
            max_counts = counts
            max_bin_x = bx
            max_bin_y = by

theta_peak = h_et_theta.GetXaxis().GetBinCenter(max_bin_x)
et_peak = h_et_theta.GetYaxis().GetBinCenter(max_bin_y)

print("\nHighest measured bin:")
print(f"(theta, et, counts) = "
      f"({theta_peak:.2f} deg, {et_peak:.2f} keV, {max_counts:.0f})")


# ============================================================
# CREATE A SMOOTH COPY
# ============================================================

old_smooth = ROOT.gROOT.FindObject("h_et_smooth")
if old_smooth:
    old_smooth.Delete()

h_et_smooth = h_et_theta.Clone("h_et_smooth")
h_et_smooth.SetDirectory(0)

# One smoothing pass
h_et_smooth.Smooth(1, "k5a")

h_et_smooth.SetTitle(
    "Smoothed Compton distribution in volume 2;"
    "Scattering angle #theta (degrees);"
    "et (keV);"
    "Smoothed counts"
)

h_et_smooth.SetStats(0)


# ============================================================
# REMOVE OLD CANVAS SAFELY
# ============================================================

old_canvas = ROOT.gROOT.FindObject("c3d_smooth")
if old_canvas:
    old_canvas.Close()


# ============================================================
# CREATE 3D CANVAS
# ============================================================

c3d_smooth = ROOT.TCanvas(
    "c3d_smooth",
    "Smoothed angle vs et vs counts",
    1100,
    800
)

c3d_smooth.SetLeftMargin(0.10)
c3d_smooth.SetRightMargin(0.14)
c3d_smooth.SetBottomMargin(0.10)

c3d_smooth.SetTheta(25)
c3d_smooth.SetPhi(35)

# Use logarithmic Z scale when requested
c3d_smooth.SetLogz(1 if use_logz else 0)

h_et_smooth.GetXaxis().SetTitleOffset(1.5)
h_et_smooth.GetYaxis().SetTitleOffset(1.6)
h_et_smooth.GetZaxis().SetTitleOffset(1.2)

# Smooth surface
h_et_smooth.Draw("SURF2Z")


# ============================================================
# MARK THE ORIGINAL MEASURED MAXIMUM
# ============================================================

peak_marker = ROOT.TPolyMarker3D(1)

peak_marker.SetPoint(
    0,
    theta_peak,
    et_peak,
    max_counts
)

peak_marker.SetMarkerStyle(29)
peak_marker.SetMarkerSize(2.5)
peak_marker.SetMarkerColor(ROOT.kRed)

peak_marker.Draw("same")


# ============================================================
# DISPLAY
# ============================================================

c3d_smooth.Modified()
c3d_smooth.Update()
c3d_smooth.Draw()

c3d_smooth

Tree entries: 100000
Selected points: 1397
de entries: 1397.0
et entries: 1397.0

Highest measured bin:
(theta, et, counts) = (21.00 deg, 52.50 keV, 18)
